<a href="https://www.kaggle.com/code/harshadamagar1/notebook6f6efee7e6?scriptVersionId=362442479" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [2]:
!apt-get update -qq && apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 125653 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!which ollama
!ollama --version

/usr/local/bin/ollama
]11;?\Warning: could not connect to a running Ollama instance


In [5]:
import subprocess
import time
import requests
import os

log_file = open("/tmp/ollama.log", "w")

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

for attempt in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=2
        )
        if response.ok:
            print("Ollama server started successfully!")
            break
    except requests.RequestException:
        time.sleep(2)
else:
    print("Ollama did not start. Check the server log:")
    print(open("/tmp/ollama.log").read())

Ollama server started successfully!


In [6]:
!ollama pull qwen3:4b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 3e4cb1417446:   1% ▕                  ▏  28 MB/2.5 GB                  pulling manifest 
pulling 3e4cb1417446:   2% ▕                  ▏  58 MB/2.5 GB                  pulling manifest 
pulling 3e4cb1417446:   3% ▕                  ▏  73 MB/2.5 GB                  pulling manifest 
pulling 3e4cb1417446:   4% ▕                  ▏ 108 MB/2.5 GB                  pulling manifest 
pulling 3e4cb1417446:   6% ▕█                 ▏ 152 MB/2.5 GB                  pulling manifest 
pulling 3e4cb1417446:   7% ▕█                 ▏ 173 MB/2.5 GB                  pulling manifest 
pulling 3e4cb1417446:   9% ▕█                 ▏ 234 MB/2.5 GB                  pulling manifest 
pulling 3e

In [7]:
%pip install -q gradio ollama

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import gradio as gr
import ollama

MODEL = "qwen3:4b"

def chat(message, history):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI tutor. Explain concepts in simple English with practical examples."
        }
    ]

    # Support older Gradio history format
    for item in history:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item

            if user_message:
                messages.append({"role": "user", "content": user_message})

            if assistant_message:
                messages.append({"role": "assistant", "content": assistant_message})

    messages.append({"role": "user", "content": message})

    response = ollama.chat(
        model=MODEL,
        messages=messages
    )

    return response["message"]["content"]


demo = gr.ChatInterface(
    fn=chat,
    title="My Qwen AI Chatbot",
    description="Ask questions and learn with your AI tutor."
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://5eacb0cba112828816.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
